In [1]:
import pandas as pd

df = pd.read_csv('/Users/tyco/Desktop/MADS/sanovo_19052025/data/processed/cleaned_data.csv')
df.head()

,MerId,MerType,MerDate,MerNumber,MerSubNumber,MerData1,MerData2,MerData3,MerData4,MerData5,MerMinId,MerText,MerDeviceType,MerDeviceNumber,MerPriority,MerDateDay
0,11897185,7,2025-01-30 10:22:00,10,22,181.0,180.0,90.0,NaN,NaN,4620,NaN,0,0,0,2025-01-30
1,11897186,21,2025-01-30 10:21:42,460,12,1.0,22.0,41.0,NaN,NaN,4620,NaN,0,0,0,2025-01-30
2,11897187,23,2025-01-30 10:21:56,460,12,0.0,0.0,0.0,NaN,NaN,4620,NaN,0,0,0,2025-01-30
3,11897188,21,2025-01-30 10:21:56,460,12,1.0,22.0,41.0,NaN,NaN,4620,NaN,0,0,0,2025-01-30
4,11897189,23,2025-01-30 10:21:57,460,12,0.0,0.0,0.0,NaN,NaN,4620,NaN,0,0,0,2025-01-30


We are going to create a window of 15 seconds and aggregate the different errors and machine stops.

In [9]:
df['MerDate'] = pd.to_datetime(df['MerDate'])
df_agg_15s = df.set_index('MerDate').resample('15S').apply(
    lambda x: pd.Series({
        'Total errors 411': ((x['MerType'] == 21) & (x['MerNumber'] == 411)).sum(),
        'Total errors 414': ((x['MerType'] == 21) & (x['MerNumber'] == 414)).sum(),
        'Total errors 460': ((x['MerType'] == 21) & (x['MerNumber'] == 460)).sum(),
        'Total errors 456': ((x['MerType'] == 21) & (x['MerNumber'] == 456)).sum(),
        'Total errors 452': ((x['MerType'] == 21) & (x['MerNumber'] == 452)).sum(),
        'Total stops': ((x['MerType'] == 8) & (x['MerNumber'] == 0)).sum()
    })
).reset_index()

df_agg_15s.head()

/var/folders/pt/00xl4yzd64s0vbx9w9tzh94m0000gn/T/ipykernel_67092/273891096.py:2: FutureWarning: 'S' is deprecated and will be removed in a future version, please use 's' instead.
  df_agg_15s = df.set_index('MerDate').resample('15S').apply(


,MerDate,Total errors 411,Total errors 414,Total errors 460,Total errors 456,Total errors 452,Total stops
0,2025-01-30 10:21:30,0,0,1,0,0,0
1,2025-01-30 10:21:45,0,0,2,0,0,0
2,2025-01-30 10:22:00,0,0,1,0,1,0
3,2025-01-30 10:22:15,1,1,1,0,0,0
4,2025-01-30 10:22:30,0,0,1,0,0,0


we are going to make a train and test set based on the ordered data from last to most recent datetime

In [6]:
df_agg_15s.sort_values('MerDate', ascending=True)
# Calculate the split index for 80% train data
split_index = int(len(df_agg_15s) * 0.8)

# Split the data into train and test sets
train = df_agg_15s.iloc[:split_index]
test = df_agg_15s.iloc[split_index:]

,MerDate,Total warnings 98,Total errors 96,Total errors 411,Total errors 414,Total errors 460,Total errors 456,Total errors 452,Total stops
0,2025-01-30 10:21:30,0,0,0,0,1,0,0,0
1,2025-01-30 10:21:45,0,0,0,0,2,0,0,0
2,2025-01-30 10:22:00,0,0,0,0,1,0,1,0
3,2025-01-30 10:22:15,1,0,1,1,1,0,0,0
4,2025-01-30 10:22:30,0,0,0,0,1,0,0,0


In [8]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

# Define features and target variable
X_train = train[['Total errors 411', 'Total errors 414', 'Total errors 460', 'Total errors 456', 'Total errors 452']]
y_train = train['Total stops']

X_test = test[['Total errors 411', 'Total errors 414', 'Total errors 460', 'Total errors 456', 'Total errors 452']]
y_test = test['Total stops']

# Initialize and train the model
model = LinearRegression()
model.fit(X_train, y_train)

# Predict on the test set
y_pred = model.predict(X_test)

# Evaluate the model
rmse = root_mean_squared_error(y_test, y_pred)
print(f"Root Mean Squared Error: {rmse}")

Root Mean Squared Error: 0.27216622600097123
